In [9]:
import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, RandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, confusion_matrix

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [10]:
SRC_NPZ = r"E:\EEG_B4\UDADataset\source_physionet_prepared.npz"
src = np.load(SRC_NPZ)
x_train_src = torch.from_numpy(src["x_train"]).float()
y_train_src = torch.from_numpy(src["y_train"]).long()
x_test_src  = torch.from_numpy(src["x_test"]).float()
y_test_src  = torch.from_numpy(src["y_test"]).long()

src_loader = DataLoader(TensorDataset(x_train_src, y_train_src),
                        batch_size=128, shuffle=True, drop_last=True,
                        num_workers=0, pin_memory=True)

# ---- Target list ----
TGT_NPZ_LIST = [
    r"E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz",
    r"E:\EEG_B4\UDADataset\P002\target_P002_S002_prepared.npz",
    r"E:\EEG_B4\UDADataset\P003\target_P003_S002_prepared.npz",
    r"E:\EEG_B4\UDADataset\P004\target_P004_S002_prepared.npz",
]
KEEP_TEST_SUBJ = "P004"
MERGE_TEST_INTO_POOL = True

x_pool_list = []
x_ft = y_ft = None
x_test = y_test = None

for p in TGT_NPZ_LIST:
    name = os.path.basename(p).split("_")[1]
    d = np.load(p)

    x_pool_i = torch.from_numpy(d["x_pool"]).float()
    x_test_i = torch.from_numpy(d["x_test"]).float()
    y_test_i = torch.from_numpy(d["y_test"]).long()

    if name == KEEP_TEST_SUBJ:
        # ---- P001: FTとTestを確定 ----
        x_ft   = torch.from_numpy(d["x_ft"]).float()
        y_ft   = torch.from_numpy(d["y_ft"]).long()
        x_test = x_test_i
        y_test = y_test_i
        x_pool_list.append(x_pool_i)     # P001 pool はUDAにも入れる
    else:
        # ---- P002-004: testも含め全部UDA用poolへ ----
        if MERGE_TEST_INTO_POOL:
            x_pool_i = torch.cat([x_pool_i, x_test_i], dim=0)
        x_pool_list.append(x_pool_i)

# UDA用pool（全員分）
x_pool = torch.cat(x_pool_list, dim=0)

# tgt_loader（個人UDAノートと同じ）
class TgtPool(Dataset):
    def __init__(self, X): self.X = X
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

Bt = 64
tgt_ds = TgtPool(x_pool)
tgt_sampler = RandomSampler(tgt_ds, replacement=True, num_samples=max(len(tgt_ds), 5000))
tgt_loader = DataLoader(tgt_ds, batch_size=Bt, sampler=tgt_sampler, drop_last=True,
                        num_workers=0, pin_memory=True)

assert x_ft is not None and y_ft is not None
assert x_test is not None and y_test is not None
C, T = x_pool.shape[1], x_pool.shape[2]

print("src:", x_train_src.shape, "tgt_pool:", x_pool.shape, "test:", x_test.shape, "ft:", x_ft.shape)

src: torch.Size([3756, 8, 1251]) tgt_pool: torch.Size([248, 8, 1251]) test: torch.Size([16, 8, 1251]) ft: torch.Size([30, 8, 1251])


In [11]:
def coral_loss(Zs, Zt, eps=1e-6):
    Zs = Zs - Zs.mean(0, keepdim=True)
    Zt = Zt - Zt.mean(0, keepdim=True)
    d = Zs.size(1)
    Cs = (Zs.T @ Zs) / (Zs.size(0)-1 + eps)
    Ct = (Zt.T @ Zt) / (Zt.size(0)-1 + eps)
    return ((Cs - Ct).pow(2).sum()) / (4.0 * d * d)

def mmd_rbf(Zs, Zt):
    with torch.no_grad():
        allz = torch.cat([Zs, Zt], 0)
        pd = torch.cdist(allz, allz, p=2)
        med = torch.median(pd[pd>0]).clamp_min(1e-6)
        gammas = [1/(2*(med**2)), 1/(2*((2*med)**2)), 1/(2*((0.5*med)**2))]
    def k(x,y,gs):
        xx=(x*x).sum(1,keepdim=True); yy=(y*y).sum(1,keepdim=True)
        dist = xx + yy.T - 2*(x@y.T)
        return sum(torch.exp(-g*dist) for g in gs)
    Kxx, Kyy, Kxy = k(Zs,Zs,gammas), k(Zt,Zt,gammas), k(Zs,Zt,gammas)
    m,n = Zs.size(0), Zt.size(0)
    return (Kxx.sum()-Kxx.diag().sum())/(m*(m-1)+1e-6) \
         + (Kyy.sum()-Kyy.diag().sum())/(n*(n-1)+1e-6) \
         - 2*Kxy.mean()


In [12]:
class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=C, samples=T):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

        # 便利: 特徴次元を属性に持っておく（UDAで使う）
        self.feature_dim = self.classify.in_features

    def _get_flattened_size(self, channels, samples):
        with torch.no_grad():
            x = torch.zeros(1, 1, channels, samples)  # (B,1,C,T)
            z = self.firstconv(x)
            z = self.depthwiseConv(z)
            z = self.separableConv(z)
            return z.numel()

    # ここから追加 ---------
    def extract_features(self, x):
        """(B,C,T) or (B,1,C,T) -> (B,d) の特徴ベクトル"""
        if x.dim() == 3:            # (B,C,T) なら (B,1,C,T) に
            x = x.unsqueeze(1)
        z = self.firstconv(x)
        z = self.depthwiseConv(z)
        z = self.separableConv(z)
        z = z.view(z.size(0), -1)   # Flatten -> (B,d)
        return z

    def forward_with_features(self, x):
        Z = self.extract_features(x)         # (B,d)
        logits = self.classify(Z)            # (B,num_classes)
        return Z, logits

    def forward(self, x):
        return self.forward_with_features(x)[1]

In [13]:
# === モデル準備 ===
model = EEGNet(num_classes=2, channels=C, samples=T).to(device)  # ←EEGNetはあなたの実装名に置換
ce = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
# 0.01 → 0.001(20ep) → 0.0001(50ep)
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[20,50], gamma=0.1)

def one_epoch(stage, lam_coral, lam_mmd):
    model.train()
    it_tgt = iter(tgt_loader)
    cls_sum = coral_sum = mmd_sum = nstep = 0.0
    for Xs, ys in src_loader:
        try:
            Xt = next(it_tgt)
        except StopIteration:
            it_tgt = iter(tgt_loader); Xt = next(it_tgt)

        # 形が(B,C,T)想定。必要なら unsqueeze(1)
        Xs, ys, Xt = Xs.to(device), ys.to(device), Xt.to(device)

        Zs, log_s = model.forward_with_features(Xs)
        Zt, _     = model.forward_with_features(Xt)

        loss_cls   = ce(log_s, ys)
        loss_coral = coral_loss(Zs, Zt) if lam_coral>0 else 0.0
        loss_mmd   = mmd_rbf(Zs, Zt)   if lam_mmd  >0 else 0.0
        loss = loss_cls + lam_coral*loss_coral + lam_mmd*loss_mmd

        opt.zero_grad(); loss.backward(); opt.step()

        cls_sum   += float(loss_cls)
        coral_sum += float(loss_coral) if lam_coral>0 else 0.0
        mmd_sum   += float(loss_mmd) if lam_mmd>0 else 0.0
        nstep     += 1
    scheduler.step()
    print(f"{stage} | CE:{cls_sum/nstep:.4f}  CORAL:{coral_sum/max(1,nstep):.4f}  MMD:{mmd_sum/max(1,nstep):.4f}")

# ----- Stage A: CORALのみ（まず安定） -----
E_A = 30
for ep in range(E_A):
    one_epoch("A", lam_coral=1e-2, lam_mmd=0.0)
torch.save(model.state_dict(), r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageA_coral.pth")

# ----- Stage B: CORAL + MMD（細部を詰める） -----
E_B = 30  # 合計60ep
for ep in range(E_B):
    one_epoch("B", lam_coral=1e-2, lam_mmd=1e-2)
torch.save(model.state_dict(), r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_coral_mmd.pth")


A | CE:0.6531  CORAL:0.0134  MMD:0.0000
A | CE:0.5654  CORAL:0.0051  MMD:0.0000
A | CE:0.5449  CORAL:0.0024  MMD:0.0000
A | CE:0.5266  CORAL:0.0016  MMD:0.0000
A | CE:0.4917  CORAL:0.0008  MMD:0.0000
A | CE:0.4876  CORAL:0.0005  MMD:0.0000
A | CE:0.4849  CORAL:0.0005  MMD:0.0000
A | CE:0.4859  CORAL:0.0004  MMD:0.0000
A | CE:0.4886  CORAL:0.0006  MMD:0.0000
A | CE:0.4740  CORAL:0.0003  MMD:0.0000
A | CE:0.4682  CORAL:0.0005  MMD:0.0000
A | CE:0.4778  CORAL:0.0003  MMD:0.0000
A | CE:0.4531  CORAL:0.0003  MMD:0.0000
A | CE:0.4485  CORAL:0.0004  MMD:0.0000
A | CE:0.4531  CORAL:0.0003  MMD:0.0000
A | CE:0.4489  CORAL:0.0002  MMD:0.0000
A | CE:0.4461  CORAL:0.0002  MMD:0.0000
A | CE:0.4359  CORAL:0.0002  MMD:0.0000
A | CE:0.4346  CORAL:0.0003  MMD:0.0000
A | CE:0.4316  CORAL:0.0002  MMD:0.0000
A | CE:0.4039  CORAL:0.0002  MMD:0.0000
A | CE:0.3998  CORAL:0.0002  MMD:0.0000
A | CE:0.4025  CORAL:0.0002  MMD:0.0000
A | CE:0.3956  CORAL:0.0002  MMD:0.0000
A | CE:0.3948  CORAL:0.0002  MMD:0.0000


In [14]:
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader

TGT_NPZ = r"E:\EEG_B4\UDADataset\P004\target_P004_S002_prepared.npz"
npz = np.load(TGT_NPZ)

# 保存済みのFTデータ（すでに x_all[ft_idx] 済み）
x_ft = torch.from_numpy(npz["x_ft"])   # (N_ft, C, T)
y_ft = torch.from_numpy(npz["y_ft"])   # (N_ft,)

# グローバル → ローカル変換に使う
ft_idx_global   = npz["ft_idx"]        # 長さ N_ft, x_all 上のインデックス（昇順のはず）
ft_train_global = npz["ft_train_idx"]  # これも x_all 上のインデックス
ft_val_global   = npz["ft_val_idx"]

# --- ここがポイント：global → local index に変換 ---
# ft_idx_global は sort 済みなので searchsorted が使える
ft_train_local = np.searchsorted(ft_idx_global, ft_train_global)
ft_val_local   = np.searchsorted(ft_idx_global, ft_val_global)

# 念のため範囲チェック
print("x_ft shape:", x_ft.shape)
print("max train_local:", ft_train_local.max(), "max val_local:", ft_val_local.max())

# これで x_ft / y_ft に安全に刺せる
x_tr, y_tr = x_ft[ft_train_local], y_ft[ft_train_local]
x_va, y_va = x_ft[ft_val_local],   y_ft[ft_val_local]

print("FT train:", x_tr.shape, "FT val:", x_va.shape)

ft_train_loader = DataLoader(TensorDataset(x_tr, y_tr),
                             batch_size=32, shuffle=True, drop_last=False)
ft_val_loader   = DataLoader(TensorDataset(x_va, y_va),
                             batch_size=64, shuffle=False, drop_last=False)


x_ft shape: torch.Size([30, 8, 1251])
max train_local: 29 max val_local: 23
FT train: torch.Size([24, 8, 1251]) FT val: torch.Size([6, 8, 1251])


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Stage B のUDA済みモデルを読み込み
model = EEGNet(num_classes=2, channels=x_pool.shape[1], samples=x_pool.shape[2]).to(device)
stageB_ckpt = r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_coral_mmd.pth"
model.load_state_dict(torch.load(stageB_ckpt, map_location=device))

# いったん全部凍結して…
for p in model.parameters():
    p.requires_grad = False

# 最終層（classify）だけ学習可にする
for p in model.classify.parameters():
    p.requires_grad = True

# Optimizer & Loss
opt = torch.optim.AdamW(model.classify.parameters(), lr=1e-2, weight_decay=1e-1)
ce  = torch.nn.CrossEntropyLoss()

# ★ここでエポック数を好きに決める
N_FT_EPOCHS = 7   # 例: 3エポック。増やしたければここを変える

for epoch in range(1, N_FT_EPOCHS + 1):
    # --- train ---
    model.train()
    tr_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in ft_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = ce(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

        tr_loss += float(loss)
        pred = logits.argmax(1)
        correct += (pred == yb).float().sum().item()
        total += len(yb)
    tr_loss /= len(ft_train_loader)
    tr_acc = correct / total if total > 0 else 0.0

    # --- val（モニタ用だけ。早期停止には使わない） ---
    model.eval()
    va_correct = 0
    va_total = 0
    with torch.no_grad():
        for xb, yb in ft_val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            va_correct += (pred == yb).float().sum().item()
            va_total += len(yb)
    va_acc = va_correct / va_total if va_total > 0 else 0.0

    print(f"[FT] epoch {epoch}  train_loss={tr_loss:.4f}  "
          f"train_acc={tr_acc*100:.1f}%  val_acc={va_acc*100:.1f}%")

# 最後のエポックのモデルを保存（必要なら）
best_path = r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_afterFT.pth"
torch.save(model.state_dict(), best_path)
print(f"Saved FT model to: {best_path}")



device: cuda
[FT] epoch 1  train_loss=0.9695  train_acc=41.7%  val_acc=66.7%
[FT] epoch 2  train_loss=0.8773  train_acc=54.2%  val_acc=66.7%
[FT] epoch 3  train_loss=0.8173  train_acc=45.8%  val_acc=66.7%
[FT] epoch 4  train_loss=0.5989  train_acc=58.3%  val_acc=50.0%
[FT] epoch 5  train_loss=0.4994  train_acc=75.0%  val_acc=50.0%
[FT] epoch 6  train_loss=0.5125  train_acc=70.8%  val_acc=50.0%
[FT] epoch 7  train_loss=0.5529  train_acc=58.3%  val_acc=50.0%
Saved FT model to: C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_afterFT.pth


In [16]:
from sklearn.metrics import f1_score, confusion_matrix

SRC_CKPT = r"C:\Users\kgtk0\Documents\Python\EEGNet_new\best.pt"

def eval_on_test(model, x_test, y_test, device):
    model.eval()
    X = x_test.to(device)
    Y = y_test.to(device)
    with torch.no_grad():
        logits = model(X)
        pred = logits.argmax(1)
    acc = (pred==Y).float().mean().item()
    f1  = f1_score(Y.cpu(), pred.cpu(), average="macro")
    cm  = confusion_matrix(Y.cpu(), pred.cpu())
    return acc, f1, cm

C, T = x_test.shape[1], x_test.shape[2]

# 1) No-UDA（ソースだけ）
model_src = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_src.load_state_dict(torch.load(SRC_CKPT, map_location=device))
acc0, f10, cm0 = eval_on_test(model_src, x_test, y_test, device)

# 3) UDA Stage B
stageB_ckpt = r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_coral_mmd.pth"
model_stageB = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_stageB.load_state_dict(torch.load(stageB_ckpt, map_location=device))
accB, f1B, cmB = eval_on_test(model_stageB, x_test, y_test, device)

# 4) UDA + FT（StageB+FT）
ckpt_stageB_FT = r"C:\Users\kgtk0\Documents\Python\UDA_all_FT\P004\ckpt_stageB_afterFT.pth"
model_stageB_FT = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_stageB_FT.load_state_dict(torch.load(ckpt_stageB_FT, map_location=device))
accBFT, f1BFT, cmBFT = eval_on_test(model_stageB_FT, x_test, y_test, device)

print(f"[No-UDA]        Acc={acc0*100:.1f}%  F1={f10:.3f}\nCM=\n{cm0}\n")
print(f"[Stage B]       Acc={accB*100:.1f}%  F1={f1B:.3f}\nCM=\n{cmB}\n")
print(f"[Stage B + FT]  Acc={accBFT*100:.1f}%  F1={f1BFT:.3f}\nCM=\n{cmBFT}\n")

print("========== Summary ==========")
print(f"No-UDA      : {acc0*100:5.1f}% (F1={f10:.3f})")
print(f"Stage B     : {accB*100:5.1f}% (F1={f1B:.3f})")
print(f"StageB + FT : {accBFT*100:5.1f}% (F1={f1BFT:.3f})")


[No-UDA]        Acc=50.0%  F1=0.492
CM=
[[3 5]
 [3 5]]

[Stage B]       Acc=56.2%  F1=0.459
CM=
[[8 0]
 [7 1]]

[Stage B + FT]  Acc=56.2%  F1=0.515
CM=
[[7 1]
 [6 2]]

========== Summary ==========
No-UDA      :  50.0% (F1=0.492)
Stage B     :  56.2% (F1=0.459)
StageB + FT :  56.2% (F1=0.515)
